# 💊 Health & Wellness Coach Agent
> Powered by **LLaMA 3.2** (Ollama Server) + OpenFDA + OpenStreetMap + DuckDuckGo

### 📋 Assignment: Build Your First AI Agent with an LLM
- **Platform:** Local LLM running on a local Ollama server
- **LLM:** LLaMA 3.2 via Ollama
- **Tools:** DuckDuckGo Search, OpenFDA API, OpenStreetMap API
- **Interface:** Gradio Web UI ⭐ (Bonus)

### 💰 Total Cost: $0
| Service | Cost |
|---------|------|
| LLaMA 3.2 (self-hosted Ollama) | ✅ Free |
| OpenFDA | ✅ Free |
| OpenStreetMap | ✅ Free |
| DuckDuckGo | ✅ Free |

---
> ⚠️ **Disclaimer:** For informational purposes only. Always consult a licensed healthcare provider.

## 📦 Step 1 — Install Dependencies

In [5]:
%pip install requests duckduckgo-search gradio -q
print('✅ All dependencies installed!')

Note: you may need to restart the kernel to use updated packages.
✅ All dependencies installed!


In [6]:
%pip install --upgrade gradio -q

Note: you may need to restart the kernel to use updated packages.


## ⚙️ Step 2 — Server Configuration
> No API key needed! Point `OLLAMA_BASE_URL` at your Ollama host.

In [ ]:
import requests

# Ollama Server
OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_MODEL    = "llama3.2"

# Test connection
try:
    resp   = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=10)
    models = [m['name'] for m in resp.json().get('models', [])]
    print(f"✅ Connected to Ollama server!")
    print(f"📦 Available models: {models}")
except Exception as e:
    print(f"❌ Cannot reach server: {e}")
    print("👉 Make sure your Ollama server is running and reachable!")

✅ Connected to Ollama server!
📦 Available models: ['Nemotron-3.5-bolt:latest', 'Qwen3.6:27B', 'qwen2.5:7b', 'x/z-image-turbo:latest', 'llava:latest', 'gemma3:12b', 'nomic-embed-text:latest', 'deepseek-r1:latest', 'llava:13b', 'llama3.2:latest', 'gemma:latest']


## 🛠️ Step 3 — Tool Implementations (All Free!)

In [8]:
import json
from duckduckgo_search import DDGS

# ─────────────────────────────────────────
# Tool 1: Symptom Checker (DuckDuckGo)
# ─────────────────────────────────────────
def check_symptoms(symptoms: str) -> dict:
    """Search trusted medical sites for symptom info."""
    try:
        query = f"{symptoms} symptoms causes treatment site:mayoclinic.org OR site:webmd.com OR site:healthline.com"
        results = []
        with DDGS() as ddgs:
            for r in ddgs.text(query, max_results=4):
                results.append({
                    "title":   r.get("title", ""),
                    "snippet": r.get("body", "")[:400],
                    "url":     r.get("href", "")
                })
        return {"symptoms": symptoms, "results": results}
    except Exception as e:
        return {"error": str(e)}


# ─────────────────────────────────────────
# Tool 2: Find Nearby Clinics (OpenStreetMap)
# ─────────────────────────────────────────
def find_nearby_clinics(location: str, clinic_type: str = "clinic") -> dict:
    """Find hospitals/clinics/pharmacies near a location."""
    try:
        geo = requests.get(
            "https://nominatim.openstreetmap.org/search",
            params={"q": location, "format": "json", "limit": 1},
            headers={"User-Agent": "HealthWellnessAgent/1.0"},
            timeout=10
        ).json()

        if not geo:
            return {"error": f"Location '{location}' not found."}

        lat = float(geo[0]["lat"])
        lon = float(geo[0]["lon"])

        amenity_map = {
            "clinic": "clinic", "hospital": "hospital",
            "pharmacy": "pharmacy", "urgent care": "clinic", "doctor": "doctors"
        }
        amenity = amenity_map.get(clinic_type.lower(), "clinic")

        overpass_query = f"""
        [out:json][timeout:10];
        node[amenity={amenity}](around:5000,{lat},{lon});
        out 5;
        """
        resp = requests.post(
            "https://overpass-api.de/api/interpreter",
            data=overpass_query, timeout=15
        ).json()

        clinics = []
        for node in resp.get("elements", [])[:5]:
            tags = node.get("tags", {})
            clinics.append({
                "name":    tags.get("name", "Unnamed Facility"),
                "address": f"{tags.get('addr:street', '')} {tags.get('addr:city', '')}".strip(),
                "phone":   tags.get("phone", "N/A"),
                "website": tags.get("website", "N/A"),
            })

        if not clinics:
            return {"location": location, "message": f"No {clinic_type}s found within 5km."}

        return {"location": location, "clinic_type": clinic_type, "clinics": clinics}
    except Exception as e:
        return {"error": str(e)}


# ─────────────────────────────────────────
# Tool 3: Medication Info (OpenFDA)
# ─────────────────────────────────────────
def get_medication_info(drug_name: str) -> dict:
    """Fetch FDA-approved drug information."""
    try:
        for field in ["openfda.brand_name", "openfda.generic_name"]:
            url  = f"https://api.fda.gov/drug/label.json?search={field}:{drug_name}&limit=1"
            resp = requests.get(url, timeout=10).json()
            if "results" in resp:
                break

        if "results" not in resp:
            return {"error": f"No FDA data found for '{drug_name}'."}

        r = resp["results"][0]
        def safe(key):
            v = r.get(key, ["N/A"])
            return v[0][:500] if v else "N/A"

        return {
            "drug":              drug_name,
            "brand_name":        r.get("openfda", {}).get("brand_name",   ["N/A"])[0],
            "generic_name":      r.get("openfda", {}).get("generic_name", ["N/A"])[0],
            "indications":       safe("indications_and_usage"),
            "warnings":          safe("warnings"),
            "dosage":            safe("dosage_and_administration"),
            "adverse_reactions": safe("adverse_reactions"),
        }
    except Exception as e:
        return {"error": str(e)}


# ─────────────────────────────────────────
# Tool 4: Wellness Tips (DuckDuckGo)
# ─────────────────────────────────────────
def get_wellness_tips(topic: str, user_profile: str = "") -> dict:
    """Search for personalized wellness and diet tips."""
    try:
        profile = f" for {user_profile}" if user_profile else ""
        query   = f"{topic} diet wellness tips{profile} site:healthline.com OR site:mayoclinic.org"
        results = []
        with DDGS() as ddgs:
            for r in ddgs.text(query, max_results=4):
                results.append({
                    "title":   r.get("title", ""),
                    "snippet": r.get("body", "")[:400],
                    "url":     r.get("href", "")
                })
        return {"topic": topic, "profile": user_profile, "results": results}
    except Exception as e:
        return {"error": str(e)}


# Tool registry
TOOL_MAP = {
    "check_symptoms":      check_symptoms,
    "find_nearby_clinics": find_nearby_clinics,
    "get_medication_info": get_medication_info,
    "get_wellness_tips":   get_wellness_tips,
}

print('✅ All 4 tools ready!')

✅ All 4 tools ready!


## 🤖 Step 4 — Agent Core (LLaMA 3.2 via Ollama)

In [9]:
SYSTEM_PROMPT = """You are a knowledgeable and compassionate AI Health & Wellness Coach.

You MUST always give helpful, detailed answers. Never refuse to answer health questions.
You are an informational assistant — not a doctor — but you CAN and SHOULD share
detailed health information, food recommendations, medication info, and wellness tips.

DECISION GUIDE — when to use tools vs answer directly:
- General health advice, food recommendations, medication info, symptoms → Answer DIRECTLY
- Need live clinic locations → use find_nearby_clinics tool
- Need official FDA drug data → use get_medication_info tool
- Need to search latest medical info → use check_symptoms or get_wellness_tips tool

WHEN USING A TOOL respond ONLY with this exact JSON (no extra text):
{"tool": "tool_name", "args": {"param1": "value1"}}

Available tools:
- check_symptoms: args = {"symptoms": "headache, fever"}
- find_nearby_clinics: args = {"location": "Wilmington NC", "clinic_type": "clinic"}
- get_medication_info: args = {"drug_name": "ibuprofen"}
- get_wellness_tips: args = {"topic": "gallstones diet", "user_profile": "adult"}

RESPONSE FORMAT:
- Use emojis and clear sections
- Use bullet points for lists
- Be warm, detailed, and specific
- For serious symptoms (chest pain, difficulty breathing) always say seek emergency care

ALWAYS end with:
⚠️ This is for informational purposes only. Please consult a licensed healthcare provider."""


def call_ollama(messages: list) -> str:
    """Send conversation to Ollama server."""
    try:
        resp = requests.post(
            f"{OLLAMA_BASE_URL}/api/chat",
            json={"model": OLLAMA_MODEL, "messages": messages, "stream": False},
            timeout=120
        )
        return resp.json()["message"]["content"]
    except Exception as e:
        return f"❌ Server error: {str(e)}"


def parse_tool_call(text: str):
    """Extract JSON tool call from LLM response."""
    try:
        start  = text.find("{")
        end    = text.rfind("}") + 1
        if start == -1 or end == 0:
            return None
        parsed = json.loads(text[start:end])
        if "tool" in parsed and "args" in parsed:
            return parsed
    except:
        pass
    return None


def run_health_agent(user_message: str, conversation_history: list = None, verbose: bool = False) -> tuple:
    """
    Run one turn of the health agent.
    verbose=False suppresses debug output for clean demo.
    Returns (response_text, updated_history)
    """
    if conversation_history is None:
        conversation_history = []

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    messages += conversation_history
    messages.append({"role": "user", "content": user_message})

    for iteration in range(6):
        llm_response = call_ollama(messages)
        tool_call    = parse_tool_call(llm_response)

        if not tool_call:
            # Final answer
            conversation_history.append({"role": "user",      "content": user_message})
            conversation_history.append({"role": "assistant", "content": llm_response})
            return llm_response, conversation_history

        # Execute tool
        tool_name = tool_call["tool"]
        tool_args = tool_call["args"]

        if verbose:
            print(f"  🔧 Calling tool: {tool_name}({tool_args})")

        fn = TOOL_MAP.get(tool_name)
        tool_result = json.dumps(fn(**tool_args) if fn else {"error": f"Unknown tool: {tool_name}"}, indent=2)

        messages.append({"role": "assistant", "content": llm_response})
        messages.append({
            "role": "user",
            "content": f"Tool '{tool_name}' returned:\n{tool_result}\n\nNow give the user a helpful, well-formatted response."
        })

    conversation_history.append({"role": "user",      "content": user_message})
    conversation_history.append({"role": "assistant", "content": llm_response})
    return llm_response, conversation_history


print(f"✅ Agent ready! Using {OLLAMA_MODEL} at {OLLAMA_BASE_URL}")

✅ Agent ready! Using llama3.2 at http://localhost:11434


In [10]:
#import sys
#!{sys.executable} -m pip uninstall gradio gradio-client -y
#!{sys.executable} -m pip install gradio==4.44.0
#print("✅ Done! Now restart kernel.")

import sys
print(sys.executable)  # Shows which Python Jupyter is using

In [11]:
#import gradio as gr
#print(gr.__version__)  # Should show 4.44.0

In [12]:
import gradio as gr

session_history = []

def chat(user_message, chat_history):
    global session_history
    if not user_message.strip():
        return "", chat_history

    response, session_history = run_health_agent(
        user_message, session_history, verbose=False
    )

    # Gradio 6.x requires dictionary format
    chat_history = chat_history + [
        {"role": "user",      "content": user_message},
        {"role": "assistant", "content": response}
    ]
    return "", chat_history


def clear_chat():
    global session_history
    session_history = []
    return [], ""


with gr.Blocks(
    title="💊 Health & Wellness Coach Agent",
    theme=gr.themes.Soft()
) as demo:

    gr.HTML("""
        <div style='text-align:center; padding:20px;
                    background: linear-gradient(135deg, #1a1a2e, #0f3460);
                    border-radius:12px; margin-bottom:16px; color:white;'>
            <h1 style='margin:0; font-size:28px;'>💊 Health & Wellness Coach Agent</h1>
            <p style='margin:8px 0 0; opacity:0.85; font-size:14px;'>
                🦙 LLaMA 3.2 via Ollama &nbsp;|&nbsp;
                OpenFDA &nbsp;|&nbsp; OpenStreetMap &nbsp;|&nbsp; DuckDuckGo
            </p>
        </div>
    """)

    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(
                label="Health Agent",
                height=480
                #type="messages"
            )
            with gr.Row():
                msg_box = gr.Textbox(
                    placeholder="Ask a health question...",
                    label="", scale=5, lines=1
                )
                send_btn = gr.Button("Send 💬", variant="primary", scale=1)
            clear_btn = gr.Button("🗑️ Clear Chat", variant="secondary")

        with gr.Column(scale=1):
            gr.HTML("""
                <div style='background:#f0f4ff; border-radius:10px; padding:16px;
                            border-left:4px solid #0f3460;'>
                    <b>💡 Example Questions</b><br><br>
                    🩺 <i>"I have fever, headache and sore throat"</i><br><br>
                    💊 <i>"Recommend a painkiller for headache"</i><br><br>
                    🏥 <i>"Find clinics near Wilmington, NC"</i><br><br>
                    🥗 <i>"Diet tips for someone with diabetes"</i><br><br>
                    💉 <i>"What are ibuprofen side effects?"</i><br><br>
                    🫀 <i>"Foods to avoid with high blood pressure"</i>
                </div>
            """)
            gr.HTML("""
                <div style='background:#fff8e1; border-radius:10px; padding:16px;
                            border-left:4px solid #f59e0b; margin-top:12px; font-size:13px;'>
                    <b>⚠️ Disclaimer</b><br>
                    General health information only.
                    Always consult a licensed healthcare provider.
                </div>
            """)

    send_btn.click(chat,  inputs=[msg_box, chatbot], outputs=[msg_box, chatbot])
    msg_box.submit(chat,  inputs=[msg_box, chatbot], outputs=[msg_box, chatbot])
    clear_btn.click(clear_chat, outputs=[chatbot, msg_box])


demo.launch(share=False, inbrowser=True)
print("🚀 Gradio app running! Open http://localhost:7860")

C:\Users\rr7756\AppData\Local\Temp\ipykernel_24336\3292370324.py:28: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


🚀 Gradio app running! Open http://localhost:7860


## 🎨 Step 5 — Gradio Web Interface (Bonus ⭐)
Run the cell below to launch a full web UI for the agent.

## 🧪 Step 6 — Quick Tests (No UI)

In [13]:
import gradio as gr
print(gr.__version__)

6.31.0


In [14]:
# Test 1: Symptom Check
response, _ = run_health_agent("I have a headache, fever, and sore throat. What could it be?")
print(response)

🤒 I'm so sorry to hear that you're not feeling well! 😷

It's always best to consult a healthcare professional for a proper diagnosis, but I can provide some general information that might help. Keep in mind that if you're experiencing severe symptoms, such as difficulty breathing, chest pain, or severe headache, you should seek emergency care immediately. 🚨

Based on your symptoms, here are a few possibilities:

* **Common Cold or Flu**: Fever, headache, and sore throat are common symptoms of a cold or flu. These viruses can cause a range of symptoms, including fatigue, congestion, and coughing.
* **Sinus Infection**: A sore throat, fever, and headache could be indicative of a sinus infection, also known as sinusitis. This infection can cause pain and pressure in the face, particularly in the cheeks and forehead.
* **Mononucleosis**: This viral infection, also known as the "kissing disease," can cause fever, sore throat, and headache. However, it's usually accompanied by other symptoms

In [15]:
# Test 2: Medication Recommendation
response, _ = run_health_agent("Recommend a painkiller for headache")
print(response)

**Painkiller Recommendation for Headache**

For headaches, I recommend using **Acetaminophen (Tylenol)** or **Ibuprofen (Advil, Motrin)**. Both are over-the-counter options that can help alleviate headache symptoms.

**Acetaminophen (Tylenol)**:
• Effective for relieving tension headaches and headaches caused by colds or flu
• Typically starts working within 30 minutes
• Recommended dose: 325-1000mg every 4-6 hours as needed

**Ibuprofen (Advil, Motrin)**:
• Also effective for relieving tension headaches and headaches caused by migraines
• May be more effective for people who experience headaches related to inflammation
• Typically starts working within 30-60 minutes
• Recommended dose: 200-400mg every 4-6 hours as needed

**Important Notes**:
• Always follow the recommended dosage and do not exceed the maximum daily dose.
• If you have a history of stomach problems, kidney disease, or allergies, consult your doctor before taking any medication.
• If your headaches are severe, frequent

In [16]:
# Test 3: Find Nearby Clinics
response, _ = run_health_agent("Find urgent care clinics near Wilmington, NC")
print(response)

📍 Urgent Care Clinics near Wilmington, NC:

• **Novant Health Urgent Care** - 4555 Carolina Beach Rd, Wilmington, NC 28403
• **Urgent Care of Wilmington** - 1555 University Blvd, Wilmington, NC 28403
• **MedExpress Urgent Care** - 2555 Market St, Wilmington, NC 28401
• **MedExpress Urgent Care** - 5555 Masonboro Rd, Wilmington, NC 28412
• **Wilmington Urgent Care** - 1340 Lumina Ave, Wilmington, NC 28405

Please note that this list may not be exhaustive and is subject to change. It's always best to check for the most up-to-date information and to find a clinic that suits your specific needs.

⚠️ This is for informational purposes only. Please consult a licensed healthcare provider.


In [17]:
# Test 4: Wellness Tips
response, _ = run_health_agent("What diet changes help with high blood pressure?")
print(response)

**Lowering Blood Pressure with Diet**

🥗 A well-planned diet can help lower blood pressure and reduce the risk of cardiovascular disease. Here are some diet changes that can help:

**Foods to Include:**

• **Leafy Greens:** Spinach, kale, collard greens, and other leafy greens are rich in potassium, calcium, and magnesium, which can help lower blood pressure.
• **Fatty Fish:** Fatty fish like salmon, tuna, and mackerel are rich in omega-3 fatty acids, which can help lower triglycerides and blood pressure.
• **Nuts and Seeds:** Almonds, pumpkin seeds, and chia seeds are rich in magnesium, potassium, and fiber, which can help lower blood pressure.
• **Beets:** Beets are rich in nitrates, which can help lower blood pressure by relaxing blood vessels.
• **Olive Oil:** Olive oil is rich in monounsaturated fats, which can help lower triglycerides and blood pressure.

**Foods to Limit or Avoid:**

• **Sodium-Rich Foods:** Processed foods, canned soups, and sauces can be high in sodium, which 

---
## 🏗️ Architecture

```
User Input (Gradio Web UI)
         │
         ▼
  LLaMA 3.2 via Ollama
  (localhost:11434)
         │
         ├──► check_symptoms()        → DuckDuckGo   (Free)
         ├──► find_nearby_clinics()   → OpenStreetMap (Free)
         ├──► get_medication_info()   → OpenFDA       (Free)
         └──► get_wellness_tips()     → DuckDuckGo   (Free)
         │
         ▼
  Structured Health Response
```

---
> ⚠️ For informational purposes only. Always consult a licensed healthcare provider.